# E-Commerce Sales Analysis Dashboard

## Overview
Comprehensive analysis of e-commerce transactions including customer purchasing patterns, product performance, and regional sales distribution.

## Dataset Information
- **Source**: workspace.default.ecommerce_large_dataset_csv
- **Period**: January - February 2025
- **Scope**: Multi-city e-commerce transactions across India

## Key Metrics
- Total Orders
- Revenue Analysis
- Product Category Performance
- Regional Distribution
- Payment Method Trends
- Order Status Tracking

In [0]:
df_clean = df.dropna()

df_clean.show()

+--------------------+
|               value|
+--------------------+
|Order_ID,Customer...|
|1001,Arjun Nair,L...|
|1002,Meera Joseph...|
|1003,Rahul Menon,...|
|1004,Anjali Das,C...|
|1005,Vinay Kumar,...|
|1006,Diya S,Lapto...|
|1007,Kiran Raj,He...|
|1008,Sneha Pillai...|
|1009,Akhil George...|
|1010,Nithin P,Pri...|
|1011,Asha Ravi,Mo...|
|1012,Rohit Das,Ke...|
|1013,Neha Thomas,...|
|1014,Aditya Menon...|
|1015,Varsha Nair,...|
|1016,Aravind K,Mo...|
|1017,Lakshmi P,He...|
|1018,Manu Raj,Pri...|
|1019,Sanjana M,Mo...|
+--------------------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col, split, trim, to_date
from pyspark.sql.types import IntegerType, DoubleType

# Parse CSV data with proper data types
df_all_parsed = df_clean \
    .filter(~col("value").contains("Order_ID,Customer_Name")) \
    .withColumn("split_col", split(col("value"), ",")) \
    .selectExpr(
        "split_col[0] as Order_ID",
        "split_col[1] as Customer_Name",
        "split_col[2] as Product",
        "split_col[3] as Category",
        "split_col[4] as Quantity",
        "split_col[5] as Price_INR",
        "split_col[6] as Total_INR",
        "split_col[7] as Order_Date",
        "split_col[8] as City",
        "split_col[9] as Payment_Method",
        "split_col[10] as Status"
    ) \
    .withColumn("Quantity", col("Quantity").cast(IntegerType())) \
    .withColumn("Price_INR", col("Price_INR").cast(DoubleType())) \
    .withColumn("Total_INR", col("Total_INR").cast(DoubleType())) \
    .withColumn("Order_Date", to_date(col("Order_Date"), "yyyy-MM-dd"))

print(f"Total Records: {df_all_parsed.count()}")
display(df_all_parsed)

Total Records: 40


Order_ID,Customer_Name,Product,Category,Quantity,Price_INR,Total_INR,Order_Date,City,Payment_Method,Status
1001,Arjun Nair,Laptop,Electronics,1,55000.0,55000.0,2025-01-01,Kochi,UPI,Delivered
1002,Meera Joseph,Keyboard,Electronics,2,1500.0,3000.0,2025-01-02,Calicut,Credit Card,Delivered
1003,Rahul Menon,Mouse,Electronics,3,700.0,2100.0,2025-01-03,Bangalore,Debit Card,Pending
1004,Anjali Das,Chair,Furniture,1,3500.0,3500.0,2025-01-04,Chennai,Cash,Delivered
1005,Vinay Kumar,Desk,Furniture,1,6000.0,6000.0,2025-01-05,Hyderabad,UPI,Cancelled
1006,Diya S,Laptop,Electronics,2,55000.0,110000.0,2025-01-06,Mumbai,Credit Card,Delivered
1007,Kiran Raj,Headphones,Electronics,4,2000.0,8000.0,2025-01-07,Delhi,UPI,Pending
1008,Sneha Pillai,Table,Furniture,1,4500.0,4500.0,2025-01-08,Pune,Debit Card,Delivered
1009,Akhil George,Monitor,Electronics,2,12000.0,24000.0,2025-01-09,Kochi,Credit Card,Delivered
1010,Nithin P,Printer,Electronics,1,8000.0,8000.0,2025-01-10,Calicut,Cash,Pending


In [0]:
from pyspark.sql.functions import sum, count, avg, col

# Payment method distribution and revenue
payment_analysis = df_all_parsed.groupBy("Payment_Method") \
    .agg(
        count("Order_ID").alias("Transaction_Count"),
        sum("Total_INR").alias("Total_Revenue"),
        avg("Total_INR").alias("Avg_Transaction_Value")
    ) \
    .orderBy(col("Transaction_Count").desc())

print("Payment Method Distribution")
print("-" * 60)
display(payment_analysis)

Payment Method Distribution
------------------------------------------------------------


Payment_Method,Transaction_Count,Total_Revenue,Avg_Transaction_Value
UPI,12,203000.0,16916.666666666668
Credit Card,12,393500.0,32791.666666666664
Debit Card,8,57900.0,7237.5
Cash,8,105000.0,13125.0


In [0]:
from pyspark.sql.functions import sum, count, col

# Order fulfillment status breakdown
status_breakdown = df_all_parsed.groupBy("Status") \
    .agg(
        count("Order_ID").alias("Order_Count"),
        sum("Total_INR").alias("Revenue")
    ) \
    .orderBy(col("Order_Count").desc())

# Calculate fulfillment rate
total_orders = df_all_parsed.count()
delivered_orders = df_all_parsed.filter(col("Status") == "Delivered").count()
fulfillment_rate = (delivered_orders / total_orders) * 100

print("Order Status Distribution")
print("-" * 60)
print(f"Overall Fulfillment Rate: {fulfillment_rate:.2f}%\n")
display(status_breakdown)

Order Status Distribution
------------------------------------------------------------
Overall Fulfillment Rate: 62.50%



Status,Order_Count,Revenue
Delivered,25,583200.0
Pending,11,130200.0
Cancelled,4,46000.0


In [0]:
from pyspark.sql.functions import col

# Example: Search for specific customer
customer_name = "Meera"  # Change this to search for any customer

customer_orders = df_all_parsed.filter(col("Customer_Name").contains(customer_name)) \
    .select("Order_ID", "Customer_Name", "Product", "Category", 
            "Quantity", "Total_INR", "Order_Date", "City", "Status")

print(f"Orders for customers matching '{customer_name}'")
print("-" * 60)
display(customer_orders)

Orders for customers matching 'Meera'
------------------------------------------------------------


Order_ID,Customer_Name,Product,Category,Quantity,Total_INR,Order_Date,City,Status
1002,Meera Joseph,Keyboard,Electronics,2,3000.0,2025-01-02,Calicut,Delivered


In [0]:
from pyspark.sql.functions import sum, avg, count, countDistinct

# Key business metrics
summary_stats = df_all_parsed.agg(
    count("Order_ID").alias("Total_Orders"),
    sum("Total_INR").alias("Total_Revenue"),
    avg("Total_INR").alias("Avg_Order_Value"),
    countDistinct("Customer_Name").alias("Unique_Customers"),
    countDistinct("Product").alias("Unique_Products"),
    countDistinct("City").alias("Cities_Served")
)

print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
display(summary_stats)

EXECUTIVE SUMMARY


Total_Orders,Total_Revenue,Avg_Order_Value,Unique_Customers,Unique_Products,Cities_Served
40,759400.0,18985.0,40,10,8


In [0]:
from pyspark.sql.functions import sum, count, avg, col

# Revenue and order count by category
category_performance = df_all_parsed.groupBy("Category") \
    .agg(
        count("Order_ID").alias("Order_Count"),
        sum("Total_INR").alias("Total_Revenue"),
        avg("Total_INR").alias("Avg_Order_Value")
    ) \
    .orderBy(col("Total_Revenue").desc())

print("Category Performance Analysis")
print("-" * 60)
display(category_performance)

Category Performance Analysis
------------------------------------------------------------


Category,Order_Count,Total_Revenue,Avg_Order_Value
Electronics,29,686900.0,23686.206896551725
Furniture,11,72500.0,6590.909090909091


In [0]:
from pyspark.sql.functions import sum, count, col

# Top 10 products by revenue
top_products = df_all_parsed.groupBy("Product", "Category") \
    .agg(
        count("Order_ID").alias("Units_Sold"),
        sum("Total_INR").alias("Total_Revenue")
    ) \
    .orderBy(col("Total_Revenue").desc()) \
    .limit(10)

print("Top 10 Products by Revenue")
print("-" * 60)
display(top_products)

Top 10 Products by Revenue
------------------------------------------------------------


Product,Category,Units_Sold,Total_Revenue
Laptop,Electronics,5,385000.0
Mobile,Electronics,4,125000.0
Monitor,Electronics,4,72000.0
Printer,Electronics,4,56000.0
Desk,Furniture,4,30000.0
Chair,Furniture,4,24500.0
Headphones,Electronics,4,24000.0
Table,Furniture,3,18000.0
Keyboard,Electronics,4,16500.0
Mouse,Electronics,4,8400.0


In [0]:
from pyspark.sql.functions import sum, count, avg, col

# Sales performance by city
regional_sales = df_all_parsed.groupBy("City") \
    .agg(
        count("Order_ID").alias("Order_Count"),
        sum("Total_INR").alias("Total_Revenue"),
        avg("Total_INR").alias("Avg_Order_Value")
    ) \
    .orderBy(col("Total_Revenue").desc())

print("Regional Sales Performance")
print("-" * 60)
display(regional_sales)

Regional Sales Performance
------------------------------------------------------------


City,Order_Count,Total_Revenue,Avg_Order_Value
Mumbai,5,195500.0,39100.0
Delhi,5,140500.0,28100.0
Bangalore,5,117100.0,23420.0
Kochi,5,98500.0,19700.0
Chennai,5,90000.0,18000.0
Hyderabad,5,47700.0,9540.0
Pune,5,35100.0,7020.0
Calicut,5,35000.0,7000.0
